# Notebook 01 — Preparación del Dataset
## Detección de Muelas del Juicio Impactadas con YOLOv8

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Repositorio:** https://github.com/JulianOliveraBalls/dentex-wisdom-teeth

---


## a) Descripción del problema y de la aplicación propuesta

### El problema clínico

Las **muelas del juicio** (terceros molares, piezas FDI 18/28/38/48) son los últimos dientes en erupcionar, típicamente entre los 17 y 25 años. Una muela del juicio **impactada** queda atrapada dentro del hueso o tejido blando, incapaz de emerger correctamente. Esto puede provocar:

- Dolor e inflamación crónica
- Infección (pericoronaritis)
- Reabsorción de los dientes adyacentes
- Quistes y daño óseo

La detección precoz en radiografías panorámicas permite planificar la extracción antes de que aparezcan complicaciones. Sin embargo, el diagnóstico manual requiere un especialista entrenado que interprete la posición angular del diente en la imagen.

### Tipo de tarea: Detección de objetos

Este proyecto aborda el problema como una tarea de **detección de objetos** (no clasificación ni segmentación):

| Tarea | Definición | ¿Es nuestro caso? |
|-------|-----------|-------------------|
| Clasificación | Asignar una etiqueta global a la imagen | No — una panorámica puede tener múltiples muelas |
| **Detección** | **Localizar objetos con bounding boxes + clase** | **Sí — detectamos cada muela individualmente** |
| Segmentación | Delinear el contorno píxel a píxel | No necesario — un bounding box es suficiente |

La salida del modelo es un conjunto de bounding boxes, cada uno con:
- Coordenadas `(x, y, w, h)` en píxeles sobre la radiografía
- Clase: `impacted` (muela retenida)
- Confianza: probabilidad del 0 al 1

### La aplicación web

El usuario interactúa con la app de la siguiente manera:

```
┌─────────────────────────────────────────────────────┐
│  [Subir radiografía panorámica]   ← input del usuario│
│                                                      │
│  [Umbral de confianza: 0.25]      ← parámetro        │
│  [SAHI (tiling): ON/OFF]          ← parámetro        │
└─────────────────────────────────────────────────────┘
                        ↓  YOLOv8m Exp_G8b
┌─────────────────────────────────────────────────────┐
│  Imagen anotada con bounding boxes                   │
│  Tabla: posición (Superior/Inferior), confianza, px  │
│  Conteo: N muelas impactadas detectadas              │
└─────────────────────────────────────────────────────┘
```

El profesional puede ajustar el umbral de confianza para balancear entre sensibilidad (no perder muelas) y especificidad (no generar falsos positivos).

### Justificación del uso de redes neuronales

¿Por qué no usar métodos clásicos (Haar cascades, HOG+SVM, umbralización morfológica)?

| Criterio | Métodos clásicos | Red neuronal (YOLOv8) |
|----------|-----------------|----------------------|
| Variabilidad de contraste | Sensibles — requieren normalización manual | Aprenden la variación internamente |
| Posiciones angulares diversas | Requieren diseñar descriptores para cada ángulo | Aprenden geometría sin ingeniería manual |
| Múltiples muelas por imagen | Requieren pipelines de sliding window ad-hoc | Detección multi-objeto nativa |
| Generalización entre equipos | Pobre — sobreajustan al tipo de aparato | Mejor con augmentation y datasets mixtos |
| Rendimiento en literatura | < 0.70 mAP50 en este tipo de problema | mAP50 = 0.992 (Exp_G8b, este proyecto) |

La naturaleza jerárquica de las panorámicas dentales (muelas con formas, orientaciones y contextos muy variables) hace que las CNNs profundas sean la elección natural: extraen características de textura, forma y contexto espacial en múltiples escalas de abstracción, algo imposible de capturar con descriptores manuales.


## b) Análisis del dataset

Este proyecto combina dos fuentes de datos públicas de radiografías panorámicas dentales con anotaciones de muelas del juicio.

---

### Dataset 1: DENTEX MICCAI 2023

| Campo | Detalle |
|-------|---------|
| **Fuente** | HuggingFace — [ibrahimhamamci/DENTEX](https://huggingface.co/datasets/ibrahimhamamci/DENTEX) |
| **Licencia** | CC BY 4.0 |
| **Contexto** | Challenge oficial MICCAI 2023 para detección de enfermedades dentales |
| **Imágenes totales** | 1.005 panorámicas (705 anotadas para enfermedades) |
| **Imágenes con muelas del juicio** | ~430 imágenes con al menos 1 muela anotada |
| **Formato de anotaciones** | COCO JSON jerárquico (cuadrante → número FDI → patología) |
| **Tamaño en disco** | ~1.8 GB (imágenes PNG) |
| **Resolución** | ~2800 × 1316 px (panorámicas de alta resolución) |
| **Relación de aspecto** | ~2.1:1 (formato panorámico estándar) |

**Estructura jerárquica de categorías COCO:**

```
category_id_1: cuadrante (Q1–Q4)
  └─ category_id_2: tipo de diente (7 = tercer molar / muela del juicio)
       └─ category_id_3: estado (0 = impacted, 1 = erupted)
```

Para este proyecto se filtra `category_id_2 == 7` (solo muelas del juicio).

---

### Dataset 2: ExAn-MTM (Expert-Annotated Mandibular Third Molar)

| Campo | Detalle |
|-------|---------|
| **Fuente labels** | Kaggle — [ikyd26/expert-annotated-mandibular-third-molar-dataset](https://www.kaggle.com/datasets/ikyd26/expert-annotated-mandibular-third-molar-dataset) |
| **Fuente imágenes** | Figshare — [DOI 10.6084/m9.figshare.21621705](https://doi.org/10.6084/m9.figshare.21621705) |
| **Licencia** | CC BY 4.0 |
| **Imágenes totales** | ~973 panorámicas |
| **Formato de anotaciones** | YOLO TXT (clase cx cy w h, normalizado) |
| **Tamaño en disco** | ~1.6 GB (imágenes JPEG) |
| **Resolución** | ~885 × 430 px (panorámicas ya reducidas) |
| **Relación de aspecto** | ~2.1:1 |

> **Nota importante:** ExAn-MTM contiene panorámicas ya en resolución reducida (~900 px de ancho), representativa de imágenes capturadas por equipos de menor resolución o transmitidas digitalmente. Esta diferencia de escala fue el insight clave para el experimento G8b.

---

### Distribución por clase (dataset combinado)

| Dataset | erupted | impacted | Total |
|---------|---------|----------|-------|
| DENTEX (con muela) | ~160 (37%) | ~270 (63%) | ~430 |
| ExAn-MTM | ~360 (37%) | ~613 (63%) | ~973 |
| **Total combinado** | **~520 (37%)** | **~883 (63%)** | **~1403** |

Existe un **desbalance moderado de 37/63%** (erupted/impacted). Este desbalance es clínicamente explicable: las muelas impactadas son más frecuentes en la población general y más relevantes diagnósticamente, por lo que los datasets tienden a sobrerrepresentarlas.

---

### Características visuales y condiciones de captura

| Característica | Descripción |
|----------------|-------------|
| **Espacio de color** | Escala de grises (imagen monocromática). Se carga como RGB con canales R=G=B. |
| **Contraste** | Variable: depende del equipo, dosis de radiación y procesado del DICOM. |
| **Ruido** | Ruido de cuantización y artefactos metálicos (obturaciones, prótesis, brackets). |
| **Fondo** | Tejido blando + hueso + aire. No hay fondo uniforme — el contexto global es parte de la imagen clínica. |
| **Resolución original** | DENTEX: ~2800px ancho. ExAn-MTM: ~885px ancho. |
| **Muelas por imagen** | Típicamente 0–4 por panorámica (una por cuadrante). |
| **Tamaño del objeto** | En panorámica DENTEX a 640px YOLO: ~20px. En imágenes reducidas (~900px): ~60px. |

**Problema de escala identificado durante el proyecto:**

Las imágenes de DENTEX (~2800px originales), al resizear internamente a 640px en YOLOv8, dejan las muelas del juicio en ~20px. Las imágenes reales (web, WhatsApp) tienen resoluciones de ~880px donde las muelas ocupan ~60px — 3x más grandes en proporción. Esta discrepancia causaba fallos en la detección de muelas **superiores** en producción. La solución fue el dataset G8b: reentrenar con las propias imágenes de DENTEX downscaleadas a ~900px, igualando la escala de entrada esperada.

---

### Particionado del dataset (modelo final G8b)

| Split | Proporción | Criterio |
|-------|-----------|----------|
| train | ~70% | Entrenamiento del modelo |
| val | ~15% | Early stopping y ajuste de hiperparámetros |
| test | ~15% | Evaluación final honesta (no visto durante entrenamiento) |

- Split aleatorio con semilla fija (`seed=42`) para reproducibilidad
- El test set nunca fue visto durante entrenamiento ni ajuste
- ExAn-MTM no aparece en el test set


---

## Sección 0 — Setup del entorno


In [ ]:
import subprocess, sys, os, gc, json, random, warnings, shutil, zipfile
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

try:
    import ultralytics, sklearn, huggingface_hub
except ImportError:
    pip_install('ultralytics', 'scikit-learn', 'huggingface_hub')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from huggingface_hub import hf_hub_download as hf_dl

# ── Utilidades ────────────────────────────────────────────────────────────────
def log(msg, level='INFO'):
    icons = {'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level, "[INFO]")} {msg}')

def is_real_file(p):
    try: return Path(p).resolve().exists() and Path(p).stat().st_size > 0
    except: return False

def split_existe(d):
    return Path(d).exists() and all(
        len(list((Path(d)/'images'/s).glob('*.*'))) > 0
        for s in ['train','val','test'])

# ── Detección automática Colab / local ────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = os.path.exists('/content') and not os.path.exists('C:/Users')
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive; drive.mount('/drive')
    REPO_ROOT = Path('/content/dentex-wisdom-teeth')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/JulianOliveraBalls/dentex-wisdom-teeth.git',
                        str(REPO_ROOT)], check=True)
    DRIVE_DIR = Path('/drive/MyDrive/dentex_runs')
else:
    _nb_path = Path(globals().get('__vsc_ipynb_file__',
                    globals().get('__file__', str(Path.cwd() / 'dev' / 'notebook.ipynb'))))
    REPO_ROOT = _nb_path.parent.parent
    DRIVE_DIR = REPO_ROOT / 'dev'
    log(f'Modo local — REPO_ROOT: {REPO_ROOT}', 'OK')

# ── Paths derivados del repo ──────────────────────────────────────────────────
DATA_DIR    = REPO_ROOT / 'data'
RAW_DIR     = DATA_DIR / 'raw'
DENTEX_DIR  = RAW_DIR / 'dentex_raw'
EXAN_DIR    = RAW_DIR / 'exan_mtm'
PROCESSED   = DATA_DIR / 'processed'
YOLO_DIR    = PROCESSED / 'yolo_dataset'
YOLO_QUAD   = PROCESSED / 'yolo_quad_flip'
YOLO_MERGED = PROCESSED / 'yolo_merged'
YOLO_G8b    = PROCESSED / 'yolo_g8b_small'
OUTPUTS_DIR = DATA_DIR / 'outputs'
SRC_DIR     = REPO_ROOT / 'src'

for d in [OUTPUTS_DIR, SRC_DIR, DENTEX_DIR, EXAN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 8
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

log(f'REPO_ROOT: {REPO_ROOT}', 'DATA')
log(f'IN_COLAB:  {IN_COLAB}   device: {device}   SEED: {SEED}', 'OK')


---

## c) Descarga y carga de los datasets

### c.1 — Dataset DENTEX (HuggingFace)

Se descarga desde HuggingFace usando `huggingface_hub`. Las anotaciones están en formato COCO JSON con jerarquía de tres niveles: cuadrante → tipo de diente → patología.


In [ ]:
import requests as _req

# ── Descarga DENTEX desde HuggingFace ────────────────────────────────────────
json_files = [f for f in DENTEX_DIR.rglob('*.json') if is_real_file(f)] if DENTEX_DIR.exists() else []

if not any('disease' in f.name.lower() for f in json_files):
    log('Descargando DENTEX desde HuggingFace...', 'WARN')
    for fname, cname in [('DENTEX/training_data.zip',   'training_data'),
                          ('DENTEX/validation_data.zip', 'validation_data')]:
        check = DENTEX_DIR / 'DENTEX' / cname
        if check.exists() and len(list(check.rglob('*.png'))) > 0:
            continue
        zp = hf_dl(repo_id='ibrahimhamamci/DENTEX', repo_type='dataset',
                   filename=fname, local_dir=str(DENTEX_DIR))
        with zipfile.ZipFile(zp, 'r') as z:
            z.extractall(str(DENTEX_DIR / 'DENTEX'))
        os.remove(zp)
        log(f'{fname} extraido', 'OK')
    json_files = [f for f in DENTEX_DIR.rglob('*.json') if is_real_file(f)]
else:
    log('DENTEX ya descargado', 'OK')

# ── Cargar anotaciones de muelas del juicio (category_id_2 == 7) ──────────────
TRAIN_JSON = next((j for j in json_files if 'disease' in j.name.lower()), None)
if not TRAIN_JSON:
    raise FileNotFoundError('JSON de DENTEX no encontrado — verificar descarga')

IMG_DIR_C = next(
    (c for c in [TRAIN_JSON.parent/'xrays', TRAIN_JSON.parent.parent/'xrays'] if c.exists()),
    TRAIN_JSON.parent / 'xrays'
)

with open(TRAIN_JSON) as f:
    data_c = json.load(f)

id2img = {img['id']: img for img in data_c['images']}
wisdom = defaultdict(list)

for ann in data_c['annotations']:
    if ann.get('category_id_2') == 7:   # solo terceros molares
        wisdom[ann['image_id']].append(ann)

def classify_image(img_id):
    return 'impacted' if any(a.get('category_id_3') == 0 for a in wisdom[img_id]) else 'erupted'

counts = Counter(classify_image(i) for i in wisdom)
total  = sum(counts.values())

log(f'DENTEX — imagenes con muela del juicio: {total}', 'DATA')
log(f'  erupted:  {counts["erupted"]}  ({counts["erupted"] / total * 100:.1f}%)', 'DATA')
log(f'  impacted: {counts["impacted"]} ({counts["impacted"] / total * 100:.1f}%)', 'DATA')
log(f'  JSON: {TRAIN_JSON.name}', 'DATA')
log(f'  Imagenes en disco: {len(list(IMG_DIR_C.glob("*.png")))} PNG', 'DATA')


### c.2 — Dataset ExAn-MTM (Kaggle + Figshare)

ExAn-MTM requiere dos descargas separadas: las **etiquetas YOLO** desde Kaggle y las **imágenes** desde Figshare (archivo ZIP de 1.6 GB). Las imágenes ya vienen en resolución reducida (~885 px de ancho).


In [ ]:
# ── Paths ExAn-MTM ────────────────────────────────────────────────────────────
EXAN_BASE      = EXAN_DIR / 'ExAn-MTM dataset'
EXAN_TRAIN_IMG = EXAN_BASE / 'train' / 'images'
EXAN_TRAIN_LBL = EXAN_BASE / 'train' / 'labels'
EXAN_VAL_IMG   = EXAN_BASE / 'valid' / 'images'
EXAN_VAL_LBL   = EXAN_BASE / 'valid' / 'labels'
for d in [EXAN_TRAIN_IMG, EXAN_VAL_IMG]:
    d.mkdir(parents=True, exist_ok=True)

# ── 1. Labels desde Kaggle ────────────────────────────────────────────────────
n_labels = len([p for p in EXAN_TRAIN_LBL.glob('*.txt')
                if p.name != 'readme.txt']) if EXAN_TRAIN_LBL.exists() else 0

if n_labels >= 875:
    log(f'ExAn-MTM labels ya descargados: {n_labels}', 'OK')
else:
    log('Descargando labels ExAn-MTM desde Kaggle...', 'WARN')
    # Requiere token Kaggle — generarlo en https://www.kaggle.com/settings/account
    KAGGLE_TOKEN = 'KAGGLE_TOKEN_AQUI'   # reemplazar con token real antes de ejecutar
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    token_path = Path.home() / '.kaggle' / 'access_token'
    token_path.parent.mkdir(exist_ok=True)
    token_path.write_text(KAGGLE_TOKEN); token_path.chmod(0o600)
    try:
        import kaggle as _kaggle
    except ImportError:
        pip_install('kaggle'); import kaggle as _kaggle
    try:
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d',
             'ikyd26/expert-annotated-mandibular-third-molar-dataset',
             '--path', str(EXAN_DIR), '--unzip'],
            capture_output=True, text=True,
            env={**os.environ, 'KAGGLE_API_TOKEN': KAGGLE_TOKEN})
        if r.returncode != 0:
            log(f'CLI fallo: {r.stderr[:120]} — intentando API Python...', 'WARN')
            import kaggle
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(
                'ikyd26/expert-annotated-mandibular-third-molar-dataset',
                path=str(EXAN_DIR), unzip=True)
        log('Labels descargados', 'OK')
    except Exception as e:
        log(f'Error Kaggle: {e}', 'ERR')
    n_labels = len([p for p in EXAN_TRAIN_LBL.glob('*.txt') if p.name != 'readme.txt'])
    log(f'Labels disponibles: {n_labels}', 'DATA')

# ── 2. Imagenes desde Figshare (1.6 GB) ──────────────────────────────────────
n_imgs = len(list(EXAN_TRAIN_IMG.glob('*.jpg')))

if n_imgs >= 875:
    log(f'ExAn-MTM imagenes ya extraidas: {n_imgs}', 'OK')
else:
    ZIP_PATH = EXAN_DIR / 'dental_dataset.zip'
    if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size < 1e9:
        log('Descargando imagenes ExAn-MTM desde Figshare (1.6 GB)...', 'WARN')
        with _req.get('https://ndownloader.figshare.com/files/38322366', stream=True) as resp:
            resp.raise_for_status()
            total_b = int(resp.headers.get('content-length', 0))
            downloaded = 0
            with open(ZIP_PATH, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    f.write(chunk); downloaded += len(chunk)
                    if downloaded % (200*1024*1024) == 0:
                        log(f'  {downloaded/1e6:.0f}/{total_b/1e6:.0f} MB', 'DATA')
        log(f'ZIP descargado: {ZIP_PATH.stat().st_size/1e9:.2f} GB', 'OK')
    else:
        log(f'ZIP ya existe: {ZIP_PATH.stat().st_size/1e9:.2f} GB', 'OK')

    log('Extrayendo imagenes...', 'WARN')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        all_names = z.namelist()
        imgs_tr  = {Path(n).stem: n for n in all_names
                    if 'Dataset and code/train/images' in n
                    and n.lower().endswith(('.jpg','.jpeg','.png'))}
        imgs_val = {Path(n).stem: n for n in all_names
                    if 'Dataset and code/test/images' in n
                    and n.lower().endswith(('.jpg','.jpeg','.png'))}
        log(f'Imgs en ZIP — train: {len(imgs_tr)}  val: {len(imgs_val)}', 'DATA')

        label_stems_tr  = [p.stem for p in EXAN_TRAIN_LBL.glob('*.txt') if p.name != 'readme.txt']
        label_stems_val = [p.stem for p in EXAN_VAL_LBL.glob('*.txt')]
        extr_tr = extr_val = 0
        for stem in label_stems_tr:
            dst = EXAN_TRAIN_IMG / f'{stem}.jpg'
            if dst.exists(): extr_tr += 1; continue
            src = imgs_tr.get(stem) or imgs_val.get(stem)
            if src: dst.write_bytes(z.read(src)); extr_tr += 1
        for stem in label_stems_val:
            dst = EXAN_VAL_IMG / f'{stem}.jpg'
            if dst.exists(): extr_val += 1; continue
            src = imgs_tr.get(stem) or imgs_val.get(stem)
            if src: dst.write_bytes(z.read(src)); extr_val += 1

    log(f'ExAn-MTM extraido: {extr_tr} train, {extr_val} val', 'OK')

log(f'ExAn-MTM final: {len(list(EXAN_TRAIN_IMG.glob("*.jpg")))} train, '
    f'{len(list(EXAN_VAL_IMG.glob("*.jpg")))} val', 'DATA')


### c.3 — Visualización de ejemplos reales del dataset

Se muestran 10 ejemplos del dataset con sus bounding boxes anotadas.

- 🟢 Verde → `erupted` (muela erupcionada normalmente)
- 🔴 Rojo → `impacted` (muela retenida/impactada)


In [ ]:
def mostrar_ejemplos_dentex(wisdom_dict, id2img_dict, img_dir, n=10, seed=42):
    ids_con_muela = list(wisdom_dict.keys())
    rng = random.Random(seed)
    rng.shuffle(ids_con_muela)
    seleccionados = ids_con_muela[:n]

    cols = 5; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 8 * rows))
    axes = axes.flatten()

    for ax_i, img_id in enumerate(seleccionados):
        info = id2img_dict[img_id]
        img_path = img_dir / Path(info['file_name']).name
        if not img_path.exists():
            axes[ax_i].set_visible(False); continue

        pil = Image.open(img_path).convert('RGB')
        W, H = pil.size
        axes[ax_i].imshow(pil, aspect='auto')

        for ann in wisdom_dict[img_id]:
            x, y, bw, bh = ann['bbox']
            cls_label = 'impacted' if ann.get('category_id_3') == 0 else 'erupted'
            color = '#FF4444' if cls_label == 'impacted' else '#44CC44'
            rect = patches.Rectangle((x, y), bw, bh,
                                      linewidth=2, edgecolor=color, facecolor='none')
            axes[ax_i].add_patch(rect)
            axes[ax_i].text(x, y - 5, cls_label, color=color, fontsize=8,
                            fontweight='bold',
                            bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))

        n_imp = sum(1 for a in wisdom_dict[img_id] if a.get('category_id_3') == 0)
        n_eru = len(wisdom_dict[img_id]) - n_imp
        axes[ax_i].set_title(
            f'{Path(info["file_name"]).stem}\n{W}x{H}px | imp={n_imp} eru={n_eru}',
            fontsize=9)
        axes[ax_i].axis('off')

    for ax_i in range(len(seleccionados), len(axes)):
        axes[ax_i].set_visible(False)

    plt.suptitle('Ejemplos DENTEX — Radiografias panoramicas con anotaciones',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'dataset_examples_dentex.png'), dpi=120, bbox_inches='tight')
    plt.show()
    log('Figura guardada en data/outputs/dataset_examples_dentex.png', 'OK')

mostrar_ejemplos_dentex(wisdom, id2img, IMG_DIR_C, n=10)


### c.4 — Estadísticas del dataset: distribución y resoluciones


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Distribucion de clases
datasets_labels = ['DENTEX\nerupted', 'DENTEX\nimpacted',
                   'ExAn-MTM\nerupted', 'ExAn-MTM\nimpacted']
valores = [counts['erupted'], counts['impacted'], 360, 613]
colores = ['#44CC44', '#FF4444', '#44CC44', '#FF4444']
bars = axes[0].bar(datasets_labels, valores, color=colores, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribucion de clases por dataset', fontweight='bold')
axes[0].set_ylabel('Imagenes')
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].set_ylim(0, max(valores) * 1.15)

# 2. Resoluciones DENTEX (muestra 50 imgs)
sample_ids = list(wisdom.keys())[:50]
widths_dentex = []; heights_dentex = []
for img_id in sample_ids:
    fname = IMG_DIR_C / Path(id2img[img_id]['file_name']).name
    if fname.exists():
        with Image.open(fname) as img:
            widths_dentex.append(img.width)
            heights_dentex.append(img.height)
axes[1].scatter(widths_dentex, heights_dentex, alpha=0.6,
                color='steelblue', edgecolors='navy', s=40)
axes[1].set_title(f'DENTEX — resoluciones (n={len(widths_dentex)})', fontweight='bold')
axes[1].set_xlabel('Ancho (px)'); axes[1].set_ylabel('Alto (px)')
if widths_dentex:
    axes[1].axvline(x=np.mean(widths_dentex), color='red', linestyle='--',
                    label=f'Media: {int(np.mean(widths_dentex))}px')
    axes[1].legend(fontsize=9)

# 3. Comparativa escala DENTEX vs ExAn-MTM vs G8b
datasets_comp = ['DENTEX\n(alta res)', 'ExAn-MTM\n(baja res)', 'G8b\n(DENTEX~900px)']
anchos  = [2800, 885, 900]
muela_px = [20, 60, 55]
x = np.arange(len(datasets_comp)); width = 0.35
b1 = axes[2].bar(x - width/2, anchos, width,
                  label='Ancho imagen (px)', color='#5588BB', edgecolor='black', linewidth=0.8)
ax2t = axes[2].twinx()
b2 = ax2t.bar(x + width/2, muela_px, width,
               label='Muela en YOLO 640px (px)', color='#FF8855', edgecolor='black', linewidth=0.8)
axes[2].set_title('Escala imagen vs tamano de muela', fontweight='bold')
axes[2].set_xticks(x); axes[2].set_xticklabels(datasets_comp, fontsize=9)
axes[2].set_ylabel('Ancho imagen (px)', color='#5588BB')
ax2t.set_ylabel('Muela en YOLO input (px)', color='#FF8855')
l1, lb1 = axes[2].get_legend_handles_labels()
l2, lb2 = ax2t.get_legend_handles_labels()
axes[2].legend(l1+l2, lb1+lb2, fontsize=8, loc='upper right')

plt.suptitle('Analisis del dataset — Distribucion y caracteristicas visuales',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'dataset_stats.png'), dpi=120, bbox_inches='tight')
plt.show()
log('Figura guardada en data/outputs/dataset_stats.png', 'OK')


---

## d) Particionado y construcción de los datasets YOLO

Se generan cuatro versiones acumulativas del dataset en formato YOLO:

```
DENTEX (308 imgs train)
    ↓  expandir_cuadrantes() — panoramica → 4 cuadrantes + flip
yolo_quad_flip (~1744 imgs train)
    ↓  construir_merged() — agrega ExAn-MTM
yolo_merged (~2619 imgs train)
    ↓  build_g8b_dataset() — solo impacted, downscale a ~900px
yolo_g8b_small (~127 imgs train)  ← dataset del modelo de produccion
```

| Dataset | Train imgs | Val | Test | nc | Uso |
|---------|-----------|-----|------|----|-----|
| `yolo_dataset` | 308 | 22 | 110 | 2 | Base DENTEX |
| `yolo_quad_flip` | ~1744 | 22 | 110 | 2 | + cuadrantes y flip |
| `yolo_merged` | ~2619 | ~120 | 110 | 2 | + ExAn-MTM |
| `yolo_g8b_small` | ~127 | ~27 | ~27 | 1 | Solo impacted, ~900px — **produccion** |


In [ ]:
from sklearn.model_selection import train_test_split

def generar_split_yolo(yolo_out, img_ids, labels, wisdom_dict, id2img_dict, img_dir):
    ids_tr, ids_tmp, _, y_tmp = train_test_split(
        img_ids, labels, test_size=0.30, stratify=labels, random_state=SEED)
    ids_val, ids_test, _, _ = train_test_split(
        ids_tmp, y_tmp, test_size=0.833, stratify=y_tmp, random_state=SEED)
    for split, ids in {'train': ids_tr, 'val': ids_val, 'test': ids_test}.items():
        (yolo_out/'images'/split).mkdir(parents=True, exist_ok=True)
        (yolo_out/'labels'/split).mkdir(parents=True, exist_ok=True)
        for img_id in ids:
            info = id2img_dict[img_id]
            src  = img_dir / Path(info['file_name']).name
            if not is_real_file(src): continue
            pil  = Image.open(src); W, H = pil.size
            dst  = yolo_out/'images'/split/src.name
            if not dst.exists(): os.symlink(src.resolve(), dst)
            lines = []
            for ann in wisdom_dict[img_id]:
                x, y, bw, bh = ann['bbox']
                xc = min(max((x + bw/2)/W, 0), 1)
                yc = min(max((y + bh/2)/H, 0), 1)
                wn = min(max(bw/W, 0), 1)
                hn = min(max(bh/H, 0), 1)
                cls = 1 if ann.get('category_id_3') == 0 else 0
                lines.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            (yolo_out/'labels'/split/(src.stem+'.txt')).write_text('\n'.join(lines))
    (yolo_out/'dataset.yaml').write_text(
        f'path: {yolo_out}\ntrain: images/train\nval:   images/val\n'
        f'test:  images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')

def expandir_cuadrantes(src, dst, overlap=0.1):
    for split in ['val','test']:
        for sub in ['images','labels']:
            d = dst/sub/split; d.mkdir(parents=True, exist_ok=True)
            for f in (src/sub/split).glob('*.*'):
                lnk = d/f.name
                if not lnk.exists(): os.symlink(f.resolve(), lnk)
    (dst/'images'/'train').mkdir(parents=True, exist_ok=True)
    (dst/'labels'/'train').mkdir(parents=True, exist_ok=True)
    n_o = n_q = 0
    for img_p in sorted((src/'images'/'train').glob('*.png')):
        lbl_p = src/'labels'/'train'/(img_p.stem+'.txt')
        if not lbl_p.exists(): continue
        pil = Image.open(img_p).convert('RGB'); W, H = pil.size
        d_o = dst/'images'/'train'/img_p.name
        if not d_o.exists(): os.symlink(img_p.resolve(), d_o)
        shutil.copy2(lbl_p, dst/'labels'/'train'/lbl_p.name); n_o += 1
        boxes = [(int(p[0]), *map(float, p[1:]))
                 for ln in lbl_p.read_text().strip().split('\n') if ln
                 for p in [ln.split()]]
        ox = int(W*overlap/2); oy = int(H*overlap/2)
        quads = [('tl',0,0,W//2+ox,H//2+oy), ('tr',W//2-ox,0,W,H//2+oy),
                 ('bl',0,H//2-oy,W//2+ox,H), ('br',W//2-ox,H//2-oy,W,H)]
        for qn, x1, y1, x2, y2 in quads:
            qW, qH = x2-x1, y2-y1
            crop = pil.crop((x1, y1, x2, y2)); qlines = []
            for cls, xc, yc, bw, bh in boxes:
                ax1=xc*W-bw*W/2; ay1=yc*H-bh*H/2
                ax2_=xc*W+bw*W/2; ay2=yc*H+bh*H/2
                ix1=max(ax1,x1)-x1; iy1=max(ay1,y1)-y1
                ix2=min(ax2_,x2)-x1; iy2=min(ay2,y2)-y1
                if ix2<=ix1 or iy2<=iy1: continue
                if (ix2-ix1)*(iy2-iy1)/(bw*W*bh*H) < 0.5: continue
                nxc=min(max((ix1+ix2)/2/qW,0),1); nyc=min(max((iy1+iy2)/2/qH,0),1)
                nw=min(max((ix2-ix1)/qW,0),1); nh=min(max((iy2-iy1)/qH,0),1)
                qlines.append(f'{cls} {nxc:.6f} {nyc:.6f} {nw:.6f} {nh:.6f}')
            if not qlines: continue
            st = f'{img_p.stem}_{qn}'
            crop.save(str(dst/'images'/'train'/(st+'.png')))
            (dst/'labels'/'train'/(st+'.txt')).write_text('\n'.join(qlines)); n_q += 1
            fl = crop.transpose(Image.FLIP_LEFT_RIGHT)
            flines = [f'{l.split()[0]} {1-float(l.split()[1]):.6f} {" ".join(l.split()[2:])}'
                      for l in qlines]
            sf = f'{img_p.stem}_{qn}_flip'
            fl.save(str(dst/'images'/'train'/(sf+'.png')))
            (dst/'labels'/'train'/(sf+'.txt')).write_text('\n'.join(flines)); n_q += 1
    (dst/'dataset.yaml').write_text(
        f'path: {dst}\ntrain: images/train\nval:   images/val\n'
        f'test:  images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')
    return n_o, n_q

def construir_merged(src_dentex, exan_ti, exan_tl, exan_vi, exan_vl, dst):
    for split in ['train','val','test']:
        (dst/'images'/split).mkdir(parents=True, exist_ok=True)
        (dst/'labels'/split).mkdir(parents=True, exist_ok=True)
    n = defaultdict(int)
    for split in ['train','val','test']:
        for img_p in sorted(list((src_dentex/'images'/split).glob('*.png')) +
                             list((src_dentex/'images'/split).glob('*.jpg'))):
            lbl_p = src_dentex/'labels'/split/(img_p.stem+'.txt')
            if not lbl_p.exists(): continue
            di = dst/'images'/split/f'dentex_{img_p.name}'
            dl = dst/'labels'/split/f'dentex_{img_p.stem}.txt'
            if not di.exists():
                real = Path(os.path.realpath(str(img_p)))
                if real.exists() and real != di: os.symlink(real, di)
                else: shutil.copy2(str(img_p), str(di))
            if not dl.exists(): shutil.copy2(str(lbl_p), str(dl))
            n[f'dentex_{split}'] += 1
    for imgs_src, lbl_src, split_dst in [
            (exan_ti, exan_tl, 'train'), (exan_vi, exan_vl, 'val')]:
        if not imgs_src or not imgs_src.exists(): continue
        for img_p in sorted(list(imgs_src.glob('*.jpg')) + list(imgs_src.glob('*.png'))):
            lbl_p = lbl_src/(img_p.stem+'.txt') if lbl_src else None
            if not lbl_p or not lbl_p.exists(): continue
            di = dst/'images'/split_dst/f'exan_{img_p.name}'
            dl = dst/'labels'/split_dst/f'exan_{img_p.stem}.txt'
            if not di.exists(): os.symlink(img_p.resolve(), di)
            if not dl.exists(): shutil.copy2(str(lbl_p), str(dl))
            n[f'exan_{split_dst}'] += 1
    (dst/'dataset.yaml').write_text(
        f'path: {dst}\ntrain: images/train\nval:   images/val\n'
        f'test:  images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')
    return dict(n)

# ── Generar datasets ──────────────────────────────────────────────────────────
if split_existe(YOLO_DIR):
    log('yolo_dataset: ya existe', 'OK')
else:
    img_ids = list(wisdom.keys())
    labels  = [classify_image(i) for i in img_ids]
    generar_split_yolo(YOLO_DIR, img_ids, labels, wisdom, id2img, IMG_DIR_C)
    log('yolo_dataset generado', 'OK')

if split_existe(YOLO_QUAD):
    log('yolo_quad_flip: ya existe', 'OK')
else:
    n_o, n_q = expandir_cuadrantes(YOLO_DIR, YOLO_QUAD)
    log(f'yolo_quad_flip: {n_o + n_q} imgs train', 'OK')

n_exan = len(list((YOLO_MERGED/'images'/'train').glob('exan_*'))) if YOLO_MERGED.exists() else 0
if n_exan >= 800:
    log(f'yolo_merged OK: {len(list((YOLO_MERGED/"images"/"train").glob("*.*")))} imgs', 'OK')
else:
    log(f'yolo_merged sin ExAn-MTM (exan={n_exan}) — reconstruyendo...', 'WARN')
    if YOLO_MERGED.exists(): shutil.rmtree(str(YOLO_MERGED))
    stats = construir_merged(YOLO_QUAD, EXAN_TRAIN_IMG, EXAN_TRAIN_LBL,
                              EXAN_VAL_IMG, EXAN_VAL_LBL, YOLO_MERGED)
    for k, v in stats.items(): log(f'  {k}: {v}', 'DATA')

log('Distribucion final yolo_merged:', 'DATA')
for split in ['train','val','test']:
    img_d = YOLO_MERGED/'images'/split; lbl_d = YOLO_MERGED/'labels'/split
    n_imgs = len(list(img_d.glob('*.*')))
    c0 = c1 = 0
    for lbl in lbl_d.glob('*.txt'):
        for line in lbl.read_text().strip().split('\n'):
            if line:
                c0 += int(line.split()[0]) == 0
                c1 += int(line.split()[0]) == 1
    log(f'  {split:5s}: {n_imgs:5d} imgs | erupted={c0} ({c0/max(c0+c1,1)*100:.0f}%) '
        f'impacted={c1} ({c1/max(c0+c1,1)*100:.0f}%)', 'DATA')


### d.2 — Construcción del dataset (modelo de producción)

El dataset resuelve el problema de escala identificado con las imágenes reales.

**Insight clave:** Las imágenes de DENTEX (~2800px) al hacer resize interno a 640px en YOLOv8 dejan las muelas en ~20px. Las imágenes reales (web/WhatsApp) son ~880px, donde las muelas ocupan ~60px — 3x más grandes. Entrenar con DENTEX downscaleado a ~900px hace que el modelo aprenda a detectar muelas del tamaño que recibirá en producción.

Características de este dataset:
- **nc=1** — solo clase `impacted`
- Solo imágenes de DENTEX con al menos 1 muela impactada
- Imágenes downscaleadas a ~900px de ancho
- Split 70/15/15 (train/val/test)


In [ ]:
def build_g8b_dataset(img_ids, img_dir, dst, target_w=900, val_ratio=0.15, test_ratio=0.15):
    ids = list(img_ids)
    random.shuffle(ids)
    n      = len(ids)
    n_test = int(n * test_ratio)
    n_val  = int(n * val_ratio)
    splits = {
        'test':  ids[:n_test],
        'val':   ids[n_test:n_test + n_val],
        'train': ids[n_test + n_val:],
    }
    stats = {}
    for split, split_ids in splits.items():
        (dst/'images'/split).mkdir(parents=True, exist_ok=True)
        (dst/'labels'/split).mkdir(parents=True, exist_ok=True)
        n_imgs = n_anns = 0
        for img_id in split_ids:
            fname    = id2img[img_id]['file_name']
            img_path = img_dir / fname
            if not img_path.exists(): continue
            pil = Image.open(img_path).convert('RGB')
            orig_w, orig_h = pil.size
            scale   = target_w / orig_w
            new_h   = int(orig_h * scale)
            pil_small = pil.resize((target_w, new_h), Image.LANCZOS)
            out_name = Path(fname).stem + '.jpg'
            pil_small.save(str(dst/'images'/split/out_name), quality=92)
            anns = [a for a in wisdom[img_id] if a.get('category_id_3') == 0]
            yolo_lines = []
            for ann in anns:
                x, y, bw, bh = ann['bbox']
                cx = min(max((x + bw/2) / orig_w, 0.0), 1.0)
                cy = min(max((y + bh/2) / orig_h, 0.0), 1.0)
                nw = min(max(bw / orig_w, 0.01), 1.0)
                nh = min(max(bh / orig_h, 0.01), 1.0)
                yolo_lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
            if not yolo_lines: continue
            (dst/'labels'/split/(Path(fname).stem+'.txt')).write_text('\n'.join(yolo_lines))
            n_imgs += 1; n_anns += len(yolo_lines)
        stats[split] = {'imgs': n_imgs, 'anns': n_anns}
        log(f'G8b {split:5s}: {n_imgs} imgs, {n_anns} anotaciones', 'DATA')
    (dst/'dataset.yaml').write_text(
        f'path: {dst}\n'
        f'train: images/train\nval: images/val\ntest: images/test\n'
        f'nc: 1\nnames:\n  0: impacted\n'
    )
    return stats

impacted_ids = [img_id for img_id, anns in wisdom.items()
                if any(a.get('category_id_3') == 0 for a in anns)]
log(f'Imagenes DENTEX con impacted: {len(impacted_ids)}', 'DATA')

if (YOLO_G8b/'dataset.yaml').exists():
    log('yolo_g8b_small: ya existe', 'OK')
else:
    log('Construyendo dataset G8b (~900px, nc=1)...', 'WARN')
    stats_g8b = build_g8b_dataset(impacted_ids, IMG_DIR_C, YOLO_G8b)
    log(f'Dataset G8b listo — train: {stats_g8b["train"]["imgs"]} imgs', 'OK')

log('', 'INFO')
log('Resumen todos los datasets:', 'DATA')
for name, path in [('yolo_dataset', YOLO_DIR), ('yolo_quad_flip', YOLO_QUAD),
                    ('yolo_merged', YOLO_MERGED), ('yolo_g8b_small', YOLO_G8b)]:
    if not path.exists(): continue
    totales = sum(len(list((path/'images'/s).glob('*.*'))) for s in ['train','val','test'])
    log(f'  {name:20s}: {totales} imgs totales', 'DATA')


---

## e) Preprocesamiento y clase Dataset

### Pipeline de preprocesamiento

YOLOv8 maneja su propio pipeline de datos internamente. La clase `DentexDataset` implementada acá es para **exploración y verificación de batches** — no es la que usa YOLOv8 durante el entrenamiento.

| Paso | Valor | Justificación |
|------|-------|---------------|
| `Resize(640, 640)` | 640×640 px | Resolución nativa de YOLOv8 para entrada |
| `ToTensor()` | float32 [0, 1] | Convierte PIL uint8 [0, 255] |
| `Normalize(ImageNet)` | mean=[0.485, 0.456, 0.406] | Backbone preentrenado con COCO/ImageNet |

**¿Por qué 640px y no 224px?** Las muelas del juicio ocupan ~30–60px en las imágenes de entrada. Con 224px quedarían en ~8px, prácticamente indetectables.

**¿Grayscale o RGB?** Las panorámicas son médicas en escala de grises. Se cargan como RGB duplicando el canal tres veces — el backbone preentrenado espera 3 canales.


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

BASE_TRANSFORM = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD)
])

MEAN_T = torch.tensor(MEAN).view(3, 1, 1)
STD_T  = torch.tensor(STD).view(3, 1, 1)


class DentexDataset(Dataset):
    CLASS_NAMES  = {0: 'erupted', 1: 'impacted'}
    CLASS_COLORS = {0: '#44CC44', 1: '#FF4444'}

    def __init__(self, split, yolo_dir, transform=None):
        self.split     = split
        self.yolo_dir  = Path(yolo_dir)
        self.transform = transform or BASE_TRANSFORM
        self.img_dir   = self.yolo_dir/'images'/split
        self.lbl_dir   = self.yolo_dir/'labels'/split
        self.samples   = [
            (p, self.lbl_dir/(p.stem+'.txt'))
            for p in sorted(list(self.img_dir.glob('*.png')) + list(self.img_dir.glob('*.jpg')))
            if (self.lbl_dir/(p.stem+'.txt')).exists()
        ]
        self.class_counts = Counter()
        for _, lp in self.samples:
            for line in lp.read_text().strip().split('\n'):
                if line: self.class_counts[int(line.split()[0])] += 1
        log(f'DentexDataset [{split}]: {len(self.samples)} imgs | '
            f'erupted={self.class_counts[0]} impacted={self.class_counts[1]}', 'OK')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        real_path = Path(os.path.realpath(str(img_path)))
        img = Image.open(real_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        boxes = []
        for line in lbl_path.read_text().strip().split('\n'):
            if line:
                cls, xc, yc, w, h = line.split()
                boxes.append([int(cls), float(xc), float(yc), float(w), float(h)])
        return img, torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 5))


log('Verificando dataset G8b...', 'INFO')
ds_g8b_train = DentexDataset('train', YOLO_G8b)
ds_g8b_val   = DentexDataset('val',   YOLO_G8b)
ds_g8b_test  = DentexDataset('test',  YOLO_G8b)


---

## f) Data augmentation

### Augmentación interna de YOLOv8 (usada en todos los experimentos)

YOLOv8 aplica augmentación de datos **internamente durante el entrenamiento**, propagando las transformaciones geométricas a los bounding boxes automáticamente. Esta es la estrategia usada en todos los experimentos, incluido el modelo de producción G8b.

Los parámetros de augmentación del experimento G8b son:

| Parámetro | Valor G8b | Descripción |
|-----------|----------|-------------|
| `fliplr` | 0.5 | Flip horizontal — simula variación de qué lado están las muelas |
| `hsv_v` | 0.4 | Variación de brillo ±40% — simula variación de exposición radiográfica |
| `hsv_s` | 0.4 | Variación de saturación ±40% — afecta contraste al convertir a RGB |
| `degrees` | 5.0 | Rotación aleatoria ±5° — simula inclinación del paciente |
| `translate` | 0.1 | Traslación aleatoria ±10% — variación en el encuadre |
| `mosaic` | 0.0 | **Desactivado** — combinar múltiples panorámicas no tiene sentido clínico |
| `mixup` | 0.0 | **Desactivado** — mezclar radiografías de pacientes distintos genera artefactos |

```python
# Parámetros de augmentación G8b (pasados directamente a model.train())
AUG_G8b = dict(
    fliplr   = 0.5,
    hsv_v    = 0.4,
    hsv_s    = 0.4,
    degrees  = 5.0,
    translate= 0.1,
    mosaic   = 0.0,
    mixup    = 0.0,
)
```

### ¿Por qué no se usó el archivo `src/augmentations_config.py`?

El repositorio incluye un archivo de configuración con 6 estrategias de augmentación (A_baseline a F_mixaug) implementadas con `torchvision.transforms`. Estas son apropiadas para tareas de **clasificación** donde la imagen se pasa a un clasificador PyTorch.

Para tareas de **detección de objetos**, la augmentación geométrica (rotación, flip, recorte) debe aplicarse **simultáneamente** a la imagen y a todos los bounding boxes. YOLOv8 implementa esto nativamente — no hay que reimplementarlo.

| Tipo de augmentación | `torchvision` (src/) | YOLOv8 interno |
|---------------------|----------------------|----------------|
| Resize + normalización | ✓ | ✓ (automático) |
| Flip horizontal | ✓ (imagen sola) | ✓ (imagen + boxes) |
| Rotación, traslación | ✓ (imagen sola) | ✓ (imagen + boxes) |
| Ajuste de brillo/contraste | ✓ | ✓ (HSV) |
| **Coordenadas de bounding boxes** | **✗ — no soportado** | **✓ — nativo** |


In [ ]:
import torchvision.transforms.functional as TF

def simular_aug_g8b(img_path, n=6):
    pil = Image.open(Path(os.path.realpath(str(img_path)))).convert('RGB')
    transforms_demo = [
        ('Original',     lambda x: x),
        ('FlipLR',        lambda x: TF.hflip(x)),
        ('Brillo +40%',   lambda x: TF.adjust_brightness(x, 1.4)),
        ('Brillo -40%',   lambda x: TF.adjust_brightness(x, 0.6)),
        ('Rotacion +5deg',lambda x: TF.rotate(x, 5)),
        ('Traslacion',    lambda x: TF.affine(x, angle=0,
                                               translate=(int(x.width*0.1), 0),
                                               scale=1.0, shear=0)),
    ]
    fig, axes = plt.subplots(1, n, figsize=(20, 4))
    for i, (name, tfm) in enumerate(transforms_demo[:n]):
        augmented = tfm(pil)
        axes[i].imshow(augmented)
        axes[i].set_title(name, fontsize=10, fontweight='bold')
        axes[i].axis('off')
    plt.suptitle(
        'Augmentaciones equivalentes a las aplicadas internamente por YOLOv8 (Exp_G8b)',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'augmentation_demo.png'), dpi=120, bbox_inches='tight')
    plt.show()
    log('Figura guardada en data/outputs/augmentation_demo.png', 'OK')

sample_imgs = list((YOLO_G8b/'images'/'train').glob('*.jpg'))
if sample_imgs:
    simular_aug_g8b(sample_imgs[0])
else:
    log('Dataset G8b no disponible — ejecutar seccion d primero', 'WARN')


---

## g) Exportar CSVs de splits


In [ ]:
def exportar_csv_splits(yolo_dir, nombre_dataset, output_dir):
    rows = []
    for split in ['train','val','test']:
        lbl_dir = yolo_dir/'labels'/split
        if not lbl_dir.exists(): continue
        for lbl_p in sorted(lbl_dir.glob('*.txt')):
            lines = [l for l in lbl_p.read_text().strip().split('\n') if l]
            clases = [int(l.split()[0]) for l in lines]
            rows.append({
                'dataset':    nombre_dataset,
                'split':      split,
                'filename':   lbl_p.stem,
                'n_boxes':    len(clases),
                'n_erupted':  clases.count(0),
                'n_impacted': clases.count(1),
            })
    df = pd.DataFrame(rows)
    csv_path = output_dir / f'{nombre_dataset}_splits.csv'
    df.to_csv(csv_path, index=False)
    log(f'{nombre_dataset}: {len(df)} registros → {csv_path.name}', 'OK')
    return df

df_merged = exportar_csv_splits(YOLO_MERGED, 'yolo_merged', OUTPUTS_DIR)
df_g8b    = exportar_csv_splits(YOLO_G8b,    'yolo_g8b',    OUTPUTS_DIR)

print('\n── Resumen yolo_merged ──')
print(df_merged.groupby('split')[['n_boxes','n_erupted','n_impacted']].sum())
print('\n── Resumen yolo_g8b ──')
print(df_g8b.groupby('split')[['n_boxes','n_impacted']].sum())


---

## Resumen del notebook

| Sección | Contenido |
|---------|----------|
| **a)** | Descripción del problema clínico, tipo de tarea (detección de objetos), interacción con la app web, justificación de redes neuronales vs. métodos clásicos |
| **b)** | Análisis de DENTEX MICCAI 2023 y ExAn-MTM: fuentes, licencias, distribución de clases, resoluciones, condiciones de captura, problema de escala |
| **c)** | Descarga automática de ambos datasets con verificación de caché y visualización de 10 ejemplos reales con estadísticas |
| **d)** | Cadena de datasets YOLO: `yolo_dataset` → `yolo_quad_flip` → `yolo_merged` → `yolo_g8b_small` |
| **e)** | Preprocesamiento (resize 640px, normalización ImageNet) y clase `DentexDataset` para exploración |
| **f)** | Augmentaciones internas de YOLOv8 usadas en todos los experimentos (G8b: fliplr, hsv_v/s, degrees, translate) y justificación de por qué no se usaron las del archivo externo |
| **g)** | Exportación de CSVs de splits para documentación y reproducibilidad |


Los notebooks de entrenamiento continúan en `dev/02_...` y subsiguientes.
